# 🌱 Smart Farm Agent System

A layered multi-agent architecture where one **Orchestrator** routes requests to three specialized sub-agents:

| Agent | Job |
|---|---|
| `QAAgent` | Translates natural-language questions → SQL → plain-English answers |
| `NarratorAgent` | Writes a scheduled daily status brief from live DB data |
| `TriageAgent` | Drafts a recommended action for a given alert |

**Two run modes — flip `MOCK_MODE` in Cell 2:**
- `MOCK_MODE = True` — runs the full routing + DB pipeline with canned LLM responses. No API key needed.
- `MOCK_MODE = False` *(default here)* — every `call_llm()` hits the real LLM API.

**Database: Microsoft SQL Server**, running locally on your PC, reached from Colab through a free **ngrok TCP tunnel**. See the setup cell right before Cell 4 for the one-time steps on your machine.

---
## Cell 1 — Install dependencies
Only needed when `MOCK_MODE = False`. Safe to run regardless.

In [ ]:
%pip install -q anthropic pymssql

In [2]:
%pip install -q anthropic flask flask-cors pyngrok

---
## Cell 2 — Configuration
Set `MOCK_MODE` and (optionally) your API key here.

In [ ]:
import os
from google.colab import userdata

# ── Toggle here ────────────────────────────────────────────────
MOCK_MODE = False         # Set False to call the real LLM API
MODEL     = "claude-sonnet-4-6"
# ───────────────────────────────────────────────────────────────

# ── SQL Server connection ────────────────────────────────────────
# Your SQL Server runs locally on your PC. Colab can't see
# "localhost", so these values come from the ngrok TCP tunnel you
# start on your machine (see the setup cell before Cell 4).
#
# Add these as Colab Secrets (🔑 icon, left sidebar):
#   SQL_HOST      → ngrok tcp hostname, e.g. "0.tcp.ngrok.io"
#   SQL_PORT      → ngrok tcp port, e.g. "12345"
#   SQL_DB        → your database name, e.g. "SmartFarmDemo"
#   SQL_USER      → a SQL Server login (SQL auth, not Windows auth)
#   SQL_PASSWORD  → that login's password
SQL_HOST     = userdata.get("SQL_HOST")
SQL_PORT     = userdata.get("SQL_PORT")
SQL_DB       = userdata.get("SQL_DB")
SQL_USER     = userdata.get("SQL_USER")
SQL_PASSWORD = userdata.get("SQL_PASSWORD")

if not MOCK_MODE:
    try:
        os.environ["Gemini API Key"] = userdata.get("Gemini_API_Key")
        print("✅ API key loaded from Colab Secrets")
    except Exception:
        print("⚠️  Could not load from Colab Secrets. Set your key manually.")

print(f"Mode : {'🟡 MOCK' if MOCK_MODE else '🟢 LIVE API'}")
print(f"Model: {MODEL}")
print(f"DB   : {SQL_DB} @ {SQL_HOST}:{SQL_PORT}")

---
## Cell 3 — LLM call wrapper
Every agent funnels through `call_llm()`. In mock mode it returns the canned response immediately; in live mode it calls the Anthropic API.

In [4]:
import requests
from google.colab import userdata
import os

# Load key once
os.environ["Gemini API Key"] = userdata.get("Gemini_API_Key")

GEMINI_MODEL = "gemini-2.5-flash"
GEMINI_URL = (
    f"https://generativelanguage.googleapis.com/v1beta/models/"
    f"{GEMINI_MODEL}:generateContent"
)

def call_llm(system: str, user: str, mock_response: str = "") -> str:
    payload = {
        "system_instruction": {"parts": [{"text": system}]},
        "contents": [{"parts": [{"text": user}]}]
    }
    response = requests.post(
        GEMINI_URL,
        headers={"Content-Type": "application/json"},
        params={"key": os.environ["Gemini API Key"]},
        json=payload
    )
    result = response.json()

    # Clear error message if something goes wrong
    if "candidates" not in result:
        raise ValueError(f"Gemini error: {result.get('error', result)}")

    return result["candidates"][0]["content"]["parts"][0]["text"]


# Sanity check
reply = call_llm(
    system="You are a helpful assistant.",
    user="Say 'Gemini wrapper OK' and nothing else.",
)
print(reply)

Gemini wrapper OK


---
## One-time setup — expose your local SQL Server to Colab

Colab runs in Google's cloud; it cannot reach `localhost` on your PC. A free **ngrok TCP tunnel** bridges the two. This is a one-time setup on your machine (repeat step 5 each session).

**On your PC:**
1. Install SQL Server Express (free) if you don't have it: https://www.microsoft.com/sql-server/sql-server-downloads
2. Open **SQL Server Configuration Manager** → *SQL Server Network Configuration* → *Protocols* → enable **TCP/IP**. Restart the SQL Server service.
3. In **SSMS**, enable *SQL Server and Windows Authentication mode* (Server Properties → Security), then create a SQL login (e.g. `farm_agent`) with a password and give it access to your database. Restart the service again.
4. Create the database in SSMS (e.g. `SmartFarmDemo`) — the notebook creates the tables inside it, you just need the empty DB to exist.
5. Install ngrok (free): https://ngrok.com/download — same tool this notebook already uses for the dashboard tunnel. Then run, in a terminal on your PC:
   ```
   ngrok config add-authtoken <your ngrok authtoken>
   ngrok tcp 1433
   ```
6. ngrok prints something like `tcp://0.tcp.ngrok.io:12345 -> localhost:1433`. That host/port is your `SQL_HOST` / `SQL_PORT`.
7. In Colab, add `SQL_HOST`, `SQL_PORT`, `SQL_DB`, `SQL_USER`, `SQL_PASSWORD` as Colab Secrets (🔑 icon, left sidebar) using those values.

**Keep the `ngrok tcp 1433` terminal window open** the whole time you run the notebook — it's the live bridge. Free ngrok reassigns the address each time you restart the tunnel, so update the `SQL_HOST` / `SQL_PORT` secrets if you restart it.

> Running a second ngrok tunnel at the same time as the dashboard's HTTP tunnel (further down) needs two tunnels active at once. If your ngrok plan complains about a simultaneous-tunnel limit, check your limits at https://dashboard.ngrok.com.

---
## Cell 4 — Demo database
Creates (or recreates) three zones, crops, sensors, readings, and alerts — now in **SQL Server** instead of SQLite.

In [ ]:
import pymssql

def get_connection():
    return pymssql.connect(
        server=SQL_HOST,
        port=str(SQL_PORT),
        user=SQL_USER,
        password=SQL_PASSWORD,
        database=SQL_DB,
    )


def run_query(sql: str) -> list:
    """Run a SELECT, return rows as a list of dicts."""
    conn = get_connection()
    cur = conn.cursor(as_dict=True)
    cur.execute(sql)
    rows = cur.fetchall()
    conn.close()
    return rows


def run_execute(sql: str, params: tuple = None):
    """Run an INSERT / UPDATE / DELETE."""
    conn = get_connection()
    cur = conn.cursor()
    if params:
        cur.execute(sql, params)
    else:
        cur.execute(sql)
    conn.commit()
    conn.close()


def setup_demo_db():
    conn = get_connection()
    cur = conn.cursor()

    ddl = [
        "IF OBJECT_ID('Recommendation','U') IS NOT NULL DROP TABLE Recommendation",
        "IF OBJECT_ID('Insight_Summary','U') IS NOT NULL DROP TABLE Insight_Summary",
        "IF OBJECT_ID('Alert','U') IS NOT NULL DROP TABLE Alert",
        "IF OBJECT_ID('SensorReading','U') IS NOT NULL DROP TABLE SensorReading",
        "IF OBJECT_ID('Sensor','U') IS NOT NULL DROP TABLE Sensor",
        "IF OBJECT_ID('Crop','U') IS NOT NULL DROP TABLE Crop",
        "IF OBJECT_ID('Zone','U') IS NOT NULL DROP TABLE Zone",
        """CREATE TABLE Zone (
             zone_id INT PRIMARY KEY, zone_name NVARCHAR(100), zone_type NVARCHAR(50))""",
        """CREATE TABLE Crop (
             crop_id INT PRIMARY KEY, zone_id INT, crop_name NVARCHAR(100),
             growth_stage NVARCHAR(50), health_score FLOAT)""",
        """CREATE TABLE Sensor (
             sensor_id INT PRIMARY KEY, zone_id INT,
             sensor_type NVARCHAR(50), status NVARCHAR(20))""",
        """CREATE TABLE SensorReading (
             reading_id INT PRIMARY KEY, sensor_id INT,
             reading_time NVARCHAR(30), temperature FLOAT,
             soil_moisture FLOAT, soil_ph FLOAT, is_anomaly INT)""",
        """CREATE TABLE Alert (
             alert_id INT PRIMARY KEY, sensor_reading_id INT, zone_id INT,
             alert_type NVARCHAR(50), severity NVARCHAR(20), status NVARCHAR(20))""",
        """CREATE TABLE Recommendation (
             rec_id INT IDENTITY(1,1) PRIMARY KEY, alert_id INT,
             suggested_action NVARCHAR(MAX), priority NVARCHAR(20))""",
        """CREATE TABLE Insight_Summary (
             summary_id INT IDENTITY(1,1) PRIMARY KEY, zone_id INT,
             summary_date NVARCHAR(20), generated_text NVARCHAR(MAX))""",
    ]
    for stmt in ddl:
        cur.execute(stmt)
    conn.commit()

    cur.executemany("INSERT INTO Zone VALUES (%s,%s,%s)", [
        (1, "Greenhouse A", "greenhouse"),
        (2, "Greenhouse B", "greenhouse"),
        (3, "Field North",  "open_field"),
    ])
    cur.executemany("INSERT INTO Crop VALUES (%s,%s,%s,%s,%s)", [
        (1, 1, "Tomato",  "flowering",  82.5),
        (2, 2, "Lettuce", "vegetative", 91.0),
        (3, 3, "Pepper",  "fruiting",   68.0),
    ])
    cur.executemany("INSERT INTO Sensor VALUES (%s,%s,%s,%s)", [
        (1, 1, "soil_moisture", "active"),
        (2, 2, "temperature",   "active"),
        (3, 3, "soil_ph",       "active"),
    ])
    cur.executemany("INSERT INTO SensorReading VALUES (%s,%s,%s,%s,%s,%s,%s)", [
        (1, 1, "2026-06-20 08:00", 24.1, 18.2, 6.1, 1),
        (2, 2, "2026-06-20 08:05", 31.4, None, None, 1),
        (3, 3, "2026-06-20 08:10", 26.0, None, 4.9,  1),
    ])
    cur.executemany("INSERT INTO Alert VALUES (%s,%s,%s,%s,%s,%s)", [
        (1, 1, 1, "low_soil_moisture", "high",   "open"),
        (2, 2, 2, "high_temperature",  "medium", "open"),
        (3, 3, 3, "low_soil_ph",       "high",   "open"),
    ])
    conn.commit()
    conn.close()


setup_demo_db()

# Preview all tables
for tbl in ["Zone", "Crop", "Sensor", "SensorReading", "Alert"]:
    rows = run_query(f"SELECT * FROM {tbl}")
    print(f"\n── {tbl} ({len(rows)} rows) ──")
    for r in rows:
        print(" ", r)

---
## Cell 5 — Agent definitions
All three sub-agents and the Orchestrator.

In [ ]:
import json
from dataclasses import dataclass
from datetime import date
from typing import Optional


SCHEMA_SUMMARY = """
Zone(zone_id, zone_name, zone_type)
Crop(crop_id, zone_id, crop_name, growth_stage, health_score)
Sensor(sensor_id, zone_id, sensor_type, status)
SensorReading(reading_id, sensor_id, reading_time, temperature,
              soil_moisture, soil_ph, is_anomaly)
Alert(alert_id, sensor_reading_id, zone_id, alert_type, severity, status)
Recommendation(rec_id, alert_id, suggested_action, priority)
"""


@dataclass
class AgentResult:
    agent:  str
    output: str
    data:   Optional[list] = None


# ── QA Agent ───────────────────────────────────────────────────
class QAAgent:
    """Natural-language question → SQL → plain-English answer."""
    name = "qa_agent"

    def handle(self, question: str) -> AgentResult:
        sql_prompt = (
            f"You are a T-SQL generator for a Microsoft SQL Server farm database.\n"
            f"Schema:\n{SCHEMA_SUMMARY}\n"
            f"Write ONE read-only T-SQL query (no semicolons, no explanation, "
            f"use TOP instead of LIMIT if you need to limit rows) "
            f"that answers: {question}"
        )
        sql = call_llm(
            system="You write precise, minimal T-SQL for SQL Server. Output only the query.",
            user=sql_prompt,
            mock_response=(
                "SELECT z.zone_name, a.alert_type, a.severity "
                "FROM Alert a JOIN Zone z ON z.zone_id = a.zone_id "
                "WHERE a.status = 'open'"
            ),
        ).strip().rstrip(";")

        try:
            rows = run_query(sql)
        except Exception as e:
            return AgentResult(self.name, f"Query failed: {e}\nSQL: {sql}")

        explain_prompt = (
            f"Question: {question}\n"
            f"Query results: {json.dumps(rows)}\n"
            f"Answer in 1-2 plain sentences."
        )
        answer = call_llm(
            system="You explain query results in clear, brief plain English.",
            user=explain_prompt,
            mock_response=(
                f"There are {len(rows)} open alerts right now, including "
                f"{', '.join(r['alert_type'] for r in rows[:3])}."
            ),
        )
        return AgentResult(self.name, answer, rows)


# ── Narrator Agent ─────────────────────────────────────────────
class NarratorAgent:
    """Writes a short status brief and saves it to Insight_Summary."""
    name = "narrator_agent"

    def handle(self, _: str = "") -> AgentResult:
        rows = run_query("""
            SELECT z.zone_name, c.crop_name, c.health_score, c.growth_stage,
                   COUNT(a.alert_id) AS open_alerts
            FROM Zone z
            LEFT JOIN Crop  c ON c.zone_id = z.zone_id
            LEFT JOIN Alert a ON a.zone_id = z.zone_id AND a.status = 'open'
            GROUP BY z.zone_id, z.zone_name, c.crop_name, c.health_score, c.growth_stage
        """)
        prompt = f"Write a 3-sentence daily farm status brief from this data: {json.dumps(rows)}"
        text = call_llm(
            system="You are an agronomy analyst writing a concise daily brief for farm managers.",
            user=prompt,
            mock_response=(
                "Greenhouse A's tomatoes are flowering at 82.5 health with one open "
                "high-severity moisture alert. Greenhouse B's lettuce is healthiest at "
                "91.0 but running warm. Field North's peppers show the lowest health "
                "score at 68.0, driven by a low soil pH alert that needs attention."
            ),
        )
        run_execute(
            "INSERT INTO Insight_Summary (zone_id, summary_date, generated_text) VALUES (%s,%s,%s)",
            (None, str(date.today()), text),
        )
        return AgentResult(self.name, text, rows)


# ── Triage Agent ───────────────────────────────────────────────
class TriageAgent:
    """Drafts a recommended action for a given alert_id."""
    name = "triage_agent"

    def handle(self, alert_id: str) -> AgentResult:
        rows = run_query(f"""
            SELECT a.alert_id, a.alert_type, a.severity,
                   z.zone_name, c.crop_name, c.health_score
            FROM Alert a
            JOIN Zone z ON z.zone_id = a.zone_id
            LEFT JOIN Crop c ON c.zone_id = z.zone_id
            WHERE a.alert_id = {int(alert_id)}
        """)
        if not rows:
            return AgentResult(self.name, f"No alert found with id {alert_id}")
        ctx = rows[0]
        prompt = (
            f"Alert context: {json.dumps(ctx)}\n"
            f"Draft a specific recommended action and a priority (low/medium/high)."
        )
        text = call_llm(
            system="You are a farm operations advisor. Be specific and actionable.",
            user=prompt,
            mock_response=(
                f"Recommended action: irrigate {ctx['zone_name']} immediately and recheck "
                f"soil moisture in 2 hours; if still low, inspect the drip line for blockage. "
                f"Priority: high."
            ),
        )
        run_execute(
            "INSERT INTO Recommendation (alert_id, suggested_action, priority) VALUES (%s,%s,%s)",
            (alert_id, text, "high"),
        )
        return AgentResult(self.name, text, rows)


# ── Orchestrator ───────────────────────────────────────────────
class Orchestrator:
    """
    Routes an incoming request to the right sub-agent.

    request_type options:
      'qa'      – natural-language question (payload = question string)
      'narrate' – daily brief (payload ignored)
      'triage'  – alert recommendation (payload = alert_id string)

    Routing is deterministic here (cheaper + more reliable).
    To add open-ended chat, insert a classification call_llm() step
    that returns one of the three keys above.
    """
    def __init__(self):
        self.agents = {
            "qa":      QAAgent(),
            "narrate": NarratorAgent(),
            "triage":  TriageAgent(),
        }

    def route(self, request_type: str, payload: str = "") -> AgentResult:
        agent = self.agents.get(request_type)
        if not agent:
            return AgentResult("orchestrator", f"Unknown request type: {request_type}")
        return agent.handle(payload)


orchestrator = Orchestrator()
print("✅ All agents and orchestrator ready.")

---
## How the HTML dashboard links to the AI agents

`smart_farm_dashboard.html` is a static page — it doesn't run any Python or talk to SQL Server directly. It's linked to the agents through the small Flask API the next cell starts:

1. The cell below spins up a **Flask server** inside this Colab session, with routes `/qa`, `/narrate`, `/triage`, `/alerts` that each call `orchestrator.route(...)` — i.e. the same agents you ran above.
2. Since Colab has no public address of its own, **ngrok** opens a second (HTTP) tunnel and prints a public URL, e.g. `https://abc123.ngrok-free.app`.
3. Open `smart_farm_dashboard.html` in your browser, paste that URL into the **API URL** field at the top, and the dashboard's JavaScript (`fetch(...)`) starts calling your live agents through that tunnel.

So there are two independent ngrok tunnels once both parts are running: one **TCP** tunnel (port 1433) carrying SQL Server traffic *into* Colab, and one **HTTP** tunnel carrying dashboard traffic *into* Colab from your browser. Re-run the cell below any time the URL goes stale (Colab restarts, ngrok session ends, etc.) and re-paste the new URL into the dashboard.

In [ ]:
import os
os.system("fuser -k 5000/tcp")
print("Port 5000 freed")

from pyngrok import ngrok
ngrok.kill()  # kills all active tunnels and the ngrok process
print("All ngrok tunnels closed")

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading
import time
from google.colab import userdata

# ── Kill any leftover port/tunnel from previous runs ──────────
import os
os.system("fuser -k 5000/tcp")
ngrok.kill()
time.sleep(1)

app = Flask(__name__)
CORS(app)

# ── Routes ────────────────────────────────────────────────────

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

@app.route("/qa", methods=["POST"])
def qa():
    body = request.get_json()
    question = body.get("question", "")
    if not question:
        return jsonify({"error": "No question provided"}), 400
    result = orchestrator.route("qa", question)
    return jsonify({"output": result.output, "data": result.data})

@app.route("/narrate", methods=["POST"])
def narrate():
    result = orchestrator.route("narrate")
    return jsonify({"output": result.output, "data": result.data})

@app.route("/triage", methods=["POST"])
def triage():
    body = request.get_json()
    alert_id = str(body.get("alert_id", "1"))
    result = orchestrator.route("triage", alert_id)
    return jsonify({"output": result.output, "data": result.data})

@app.route("/alerts", methods=["GET"])
def get_alerts():
    rows = run_query("""
        SELECT a.alert_id, a.alert_type, a.severity, a.status, z.zone_name
        FROM Alert a JOIN Zone z ON z.zone_id = a.zone_id
    """)
    return jsonify(rows)

# ── Start Flask then ngrok ────────────────────────────────────
flask_thread = threading.Thread(
    target=lambda: app.run(port=5000, use_reloader=False, debug=False)
)
flask_thread.daemon = True
flask_thread.start()
time.sleep(2)

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))
public_url = ngrok.connect(5000).public_url

print(f"✅ Public URL: {public_url}")
print(f"\n   Paste in dashboard: {public_url}")
print(f"   Health check:       {public_url}/health")

In [ ]:
import os
os.system("fuser -k 5000/tcp")
print("Port 5000 freed")

---
## Cell 6 — QA Agent
Ask any natural-language question about the farm database.

In [ ]:
question = "What alerts are currently open?"   # ← change me

result = orchestrator.route("qa", question)

print("Answer")
print("─" * 50)
print(result.output)

print("\nRaw rows returned by the query")
print("─" * 50)
for row in (result.data or []):
    print(" ", row)

---
## Cell 7 — Narrator Agent
Generates and saves a daily farm status brief.

In [ ]:
result = orchestrator.route("narrate")

print("Daily Brief")
print("─" * 50)
print(result.output)

print("\nBrief saved to Insight_Summary:")
for row in run_query("SELECT * FROM Insight_Summary"):
    print(" ", row)

---
## Cell 8 — Triage Agent
Drafts an action plan for a specific alert. Try alert IDs 1, 2, or 3.

In [ ]:
alert_id = "2"   # ← try 1, 2, or 3

result = orchestrator.route("triage", alert_id)

print(f"Triage — Alert #{alert_id}")
print("─" * 50)
print(result.output)

print("\nAlert context used")
print("─" * 50)
for row in (result.data or []):
    print(" ", row)

print("\nAll saved recommendations:")
for row in run_query("SELECT * FROM Recommendation"):
    print(" ", row)

---
## Cell 9 — Run all agents end-to-end
Resets the DB and fires all three routes in sequence, the same way the original `__main__` block does.

In [ ]:
# Fresh DB so Recommendation / Insight_Summary rows don't stack up
setup_demo_db()

scenarios = [
    ("qa",      "What alerts are currently open?"),
    ("narrate", ""),
    ("triage",  "1"),
    ("triage",  "2"),
    ("triage",  "3"),
]

for req_type, payload in scenarios:
    label = f"{req_type.upper()} — {payload or '(no payload)'}"
    print("\n" + "=" * 60)
    print(label)
    print("=" * 60)
    r = orchestrator.route(req_type, payload)
    print(r.output)